# MiniGPT from Scratch — Google Colab (GPU)

This notebook trains a GPT-style language model entirely from scratch on Shakespeare's works.

**Runtime**: Set to GPU (Runtime → Change runtime type → T4 GPU) for ~5-15 min training.  
CPU training takes ~30-90 minutes.

What this builds:
- Real BPE tokenizer (same algorithm as GPT-2's tokenizer)
- Causal multi-head self-attention (from scratch, manually)
- Full GPT decoder-only transformer (4 layers, 256 dims)
- Training loop with loss curves
- Text generation with greedy and temperature sampling

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────
!pip install -q torch==2.3.1 regex==2024.5.15 matplotlib==3.8.4 tqdm==4.66.4

import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', 'GPU ✓' if torch.cuda.is_available() else 'CPU (consider switching to GPU)')
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ── Download Shakespeare corpus ──────────────────────────────────────────
import os, urllib.request
os.makedirs('data', exist_ok=True)
os.makedirs('results/checkpoints', exist_ok=True)
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
urllib.request.urlretrieve(url, 'data/corpus.txt')
with open('data/corpus.txt') as f:
    text = f.read()
print(f'Corpus: {len(text):,} characters')
print('\nFirst 300 characters:')
print(text[:300])

In [ ]:
# ── BPE Tokenizer (from scratch) ─────────────────────────────────────────
import collections, json, regex

GPT2_PAT = regex.compile(
    r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)

def text_to_byte_tokens(text):
    words = regex.findall(GPT2_PAT, text)
    return [list(w.encode('utf-8')) for w in words]

def get_pair_counts(vocab_tokens):
    counts = collections.defaultdict(int)
    for seq in vocab_tokens:
        for a, b in zip(seq, seq[1:]):
            counts[(a, b)] += 1
    return counts

def merge_pair(vocab_tokens, pair, new_id):
    a, b = pair
    result = []
    for seq in vocab_tokens:
        new_seq, i = [], 0
        while i < len(seq):
            if i < len(seq)-1 and seq[i] == a and seq[i+1] == b:
                new_seq.append(new_id); i += 2
            else:
                new_seq.append(seq[i]); i += 1
        result.append(new_seq)
    return result

def apply_merge(ids, a, b, new_id):
    result, i = [], 0
    while i < len(ids):
        if i < len(ids)-1 and ids[i] == a and ids[i+1] == b:
            result.append(new_id); i += 2
        else:
            result.append(ids[i]); i += 1
    return result

class BPETokenizer:
    def __init__(self): self.vocab, self.merges = {}, []
    
    def train(self, text, num_merges=500, verbose=True):
        self.vocab = {i: bytes([i]) for i in range(256)}
        self.merges = []
        vocab_tokens = text_to_byte_tokens(text)
        print(f'Training BPE: {len(vocab_tokens)} word-pieces, {num_merges} merges...')
        for i in range(num_merges):
            counts = get_pair_counts(vocab_tokens)
            if not counts: break
            best = max(counts, key=lambda p: (counts[p], p))
            if counts[best] < 2: break
            new_id = 256 + i
            self.merges.append(best)
            self.vocab[new_id] = self.vocab[best[0]] + self.vocab[best[1]]
            vocab_tokens = merge_pair(vocab_tokens, best, new_id)
            if verbose and (i+1) % 100 == 0:
                print(f'  merge {i+1}/{num_merges}: {self.vocab[best[0]]!r}+{self.vocab[best[1]]!r} → {new_id}')
        print(f'Vocab size: {len(self.vocab)}')
    
    def encode(self, text):
        words = regex.findall(GPT2_PAT, text)
        all_ids = []
        for word in words:
            ids = list(word.encode('utf-8'))
            for (a,b), new_id in zip(self.merges, range(256, 256+len(self.merges))):
                ids = apply_merge(ids, a, b, new_id)
            all_ids.extend(ids)
        return all_ids
    
    def decode(self, ids):
        return b''.join(self.vocab[i] for i in ids).decode('utf-8', errors='replace')
    
    @property
    def vocab_size(self): return len(self.vocab)

tokenizer = BPETokenizer()
tokenizer.train(text, num_merges=500)

# Quick round-trip test
test = 'To be, or not to be!'
assert tokenizer.decode(tokenizer.encode(test)) == test, 'Round-trip FAILED'
print(f'Round-trip test: PASSED ✓')
print(f'Vocab size: {tokenizer.vocab_size}')

In [ ]:
# ── Model: Attention, Transformer Block, MiniGPT ─────────────────────────
import math
import torch, torch.nn as nn, torch.nn.functional as F
from dataclasses import dataclass

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, max_seq_len, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model; self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len))
        self.register_buffer('mask', mask.view(1,1,max_seq_len,max_seq_len))
    
    def forward(self, x):
        B, T, C = x.shape
        def heads(t): return t.view(B,T,self.num_heads,self.d_head).transpose(1,2)
        Q, K, V = heads(self.W_Q(x)), heads(self.W_K(x)), heads(self.W_V(x))
        scores = (Q @ K.transpose(-2,-1)) / math.sqrt(self.d_head)
        scores = scores.masked_fill(self.mask[:,:,:T,:T]==0, float('-inf'))
        w = self.attn_drop(F.softmax(scores, dim=-1))
        out = (w @ V).transpose(1,2).contiguous().view(B,T,C)
        return self.proj_drop(self.W_O(out))

class FFN(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model,4*d_model), nn.GELU(),
                                  nn.Linear(4*d_model,d_model), nn.Dropout(dropout))
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, d_model, num_heads, max_seq_len, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, num_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FFN(d_model, dropout)
    def forward(self, x): return x + self.ffn(self.ln2(x + self.attn(self.ln1(x))))

@dataclass
class GPTConfig:
    vocab_size: int = 1000; d_model: int = 256; num_heads: int = 8
    num_layers: int = 4; max_seq_len: int = 256; dropout: float = 0.1

class MiniGPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg.d_model, cfg.num_heads, cfg.max_seq_len, cfg.dropout) for _ in range(cfg.num_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init)
        print(f'MiniGPT: {sum(p.numel() for p in self.parameters()):,} parameters')
    
    def _init(self, m):
        if isinstance(m, nn.Linear): nn.init.normal_(m.weight, 0, 0.02)
        elif isinstance(m, nn.Embedding): nn.init.normal_(m.weight, 0, 0.02)
        elif isinstance(m, nn.LayerNorm): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
    
    def forward(self, ids, targets=None):
        B, T = ids.shape
        pos = torch.arange(T, device=ids.device)
        x = self.drop(self.tok_emb(ids) + self.pos_emb(pos))
        for block in self.blocks: x = block(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss
    
    @torch.no_grad()
    def generate(self, ids, max_new, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new):
            ctx = ids[:, -self.cfg.max_seq_len:]
            logits, _ = self(ctx)
            logits = logits[:, -1, :]
            if temperature <= 0:
                nxt = logits.argmax(-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k:
                    v = logits.topk(top_k).values[:, -1, None]
                    logits = logits.masked_fill(logits < v, float('-inf'))
                nxt = torch.multinomial(F.softmax(logits, -1), 1)
            ids = torch.cat([ids, nxt], 1)
        return ids

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────
import time
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

BATCH_SIZE = 64; SEQ_LEN = 256; EPOCHS = 10; LR = 3e-4
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class TextDataset(Dataset):
    def __init__(self, ids, seq_len):
        self.data = torch.tensor(ids, dtype=torch.long)
        self.n = max(0, len(ids)-seq_len)
        self.seq_len = seq_len
    def __len__(self): return self.n
    def __getitem__(self, i):
        c = self.data[i:i+self.seq_len+1]; return c[:-1], c[1:]

print('Tokenizing...')
all_ids = tokenizer.encode(text)
print(f'{len(all_ids):,} tokens')

split = int(len(all_ids)*0.9)
train_loader = DataLoader(TextDataset(all_ids[:split], SEQ_LEN), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TextDataset(all_ids[split:], SEQ_LEN), batch_size=BATCH_SIZE)

cfg = GPTConfig(vocab_size=tokenizer.vocab_size)
model = MiniGPT(cfg).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR)
scaler = torch.cuda.amp.GradScaler() if device.type=='cuda' else None

train_losses, val_losses = [], []
print(f'\nTraining on {device}...')

for epoch in range(1, EPOCHS+1):
    model.train(); tl, n = 0, 0
    for x, y in tqdm(train_loader, desc=f'Epoch {epoch}'):
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        if scaler:
            with torch.cuda.amp.autocast(): _, loss = model(x, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        else:
            _, loss = model(x, y); loss.backward(); opt.step()
        tl += loss.item(); n += 1
    train_loss = tl/n
    
    model.eval(); vl, vn = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            _, loss = model(x.to(device), y.to(device))
            vl += loss.item(); vn += 1
    val_loss = vl/vn
    
    train_losses.append(train_loss); val_losses.append(val_loss)
    print(f'Epoch {epoch}: train={train_loss:.4f}  val={val_loss:.4f}')

torch.save({'model_state_dict': model.state_dict(), 'config': cfg}, 'results/checkpoints/best_model.pt')
print('Model saved!')

plt.figure(figsize=(10,5))
plt.plot(train_losses, label='Train'); plt.plot(val_losses, label='Val')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Training Curves')
plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
# ── Generation ────────────────────────────────────────────────────────────
def generate(prompt, max_tokens=200, temperature=0.9, top_k=50):
    ids = tokenizer.encode(prompt) or [0]
    t = torch.tensor([ids], dtype=torch.long, device=device)
    out = model.generate(t, max_tokens, temperature=temperature, top_k=top_k)
    return tokenizer.decode(out[0].tolist())

print('=== GREEDY DECODING ===')
print(generate('HAMLET:', max_tokens=150, temperature=0.0))

print('\n=== TEMPERATURE SAMPLING (T=0.9, top_k=50) ===')
print(generate('HAMLET:', max_tokens=150, temperature=0.9, top_k=50))

print('\n=== HIGH TEMPERATURE (T=1.4) ===')
print(generate('To be or not', max_tokens=150, temperature=1.4, top_k=50))

In [ ]:
# ── Perplexity ────────────────────────────────────────────────────────────
import math

model.eval(); total, n = 0, 0
with torch.no_grad():
    for x, y in val_loader:
        _, loss = model(x.to(device), y.to(device))
        total += loss.item() * y.numel(); n += y.numel()
ppl = math.exp(total/n)
print(f'Validation Perplexity: {ppl:.2f}')
print(f'Random baseline (vocab={tokenizer.vocab_size}): {tokenizer.vocab_size}')
print(f'Improvement ratio: {tokenizer.vocab_size/ppl:.1f}x better than random')